# 🎲 Leitor de Livros de RPG com LlamaIndex (RAG)

Este notebook:
1. Lê um PDF (livros de até ~350 páginas)
2. Limpa o texto e **quebra em vários trechos (chunks)** mantendo o número da página
3. Cria um **índice vetorial** (salvo localmente na sessão do Colab, sem precisar do Google Drive)
4. Responde perguntas citando as páginas usadas, em modo pergunta única ou **chat com memória**

**LLM e embeddings:** Google Gemini (tem camada gratuita).

> ⚙️ Antes de começar: no Colab, clique no ícone de 🔑 **Secrets** (barra lateral esquerda) e crie um segredo chamado `GOOGLE_API_KEY` com sua chave do [Google AI Studio](https://aistudio.google.com/apikey). Ative o acesso do notebook ao segredo.

## 1. Instalação das dependências

In [5]:
!pip install -q llama-index llama-index-llms-google-genai llama-index-embeddings-huggingface sentence-transformers llama-index-readers-file pymupdf

In [6]:
import os
import nest_asyncio
import re
from google import genai
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from google.colab import userdata
from pathlib import Path
from llama_index.core import Document
from llama_index.readers.file import PyMuPDFReader
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.core import PromptTemplate
from llama_index.core.postprocessor import SimilarityPostprocessor
from IPython.display import Markdown, display

In [9]:
import os
from google import genai
from google.colab import userdata

# Obtém a chave diretamente do Secrets do Colab ou do ambiente
api_key = os.environ.get('GOOGLE_API_KEY') or userdata.get('GOOGLE_API_KEY')

if not api_key:
    print("❌ Chave GOOGLE_API_KEY não configurada nos Secrets do Colab!")
else:
    client = genai.Client(api_key=api_key)
    print("📋 Modelos disponíveis na sua chave:")
    for m in client.models.list():
        if "generateContent" in (m.supported_actions or []):
            print("  -", m.name.replace("models/", ""))

📋 Modelos disponíveis na sua chave:
  - gemini-2.5-flash
  - gemini-2.5-pro
  - gemini-2.5-flash-preview-tts
  - gemini-2.5-pro-preview-tts
  - gemma-4-26b-a4b-it
  - gemma-4-31b-it
  - gemini-flash-latest
  - gemini-flash-lite-latest
  - gemini-pro-latest
  - gemini-2.5-flash-lite
  - gemini-2.5-flash-image
  - gemini-3-flash-preview
  - gemini-3.1-pro-preview
  - gemini-3.1-pro-preview-customtools
  - gemini-3.1-flash-lite-preview
  - gemini-3.1-flash-lite
  - gemini-3-pro-image-preview
  - gemini-3-pro-image
  - nano-banana-pro-preview
  - gemini-3.1-flash-image-preview
  - gemini-3.1-flash-image
  - gemini-3.1-flash-lite-image
  - gemini-nano-banana-2.1
  - gemini-3.5-flash
  - gemini-3.5-flash-lite
  - gemini-omni-flash-preview
  - gemini-omni-1.1-flash
  - gemini-omni-1.2-flash
  - gemini-3.5-transcribe
  - gemini-3.6-flash
  - gemini-3.7-flash
  - gemini-3.8-flash
  - gemini-4-argon
  - lyria-3-clip-preview
  - lyria-3-pro-preview
  - lyria-3.5
  - gemini-3.1-flash-tts-preview
 

## 2. Configurações
Ajuste dos modelos e o tamanho dos trechos.

In [10]:
# Ativa suporte a chamadas assíncronas no Colab
nest_asyncio.apply()

# Chave da API do Google AI Studio
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

# --- Configurações Otimizadas ---
LLM_MODEL = 'gemini-3.8-flash-lite-tts'
CHUNK_SIZE = 500               # Trechos mais leves e objetivos
CHUNK_OVERLAP = 50             # Pequena margem de segurança
TOP_K = 10                      # Quantidade de trechos consultados por pergunta
PERSIST_DIR = '/content/rpg_indices'

Settings.llm = GoogleGenAI(model=LLM_MODEL, temperature=0.2)

# HuggingFace (SEM LIMITE DE COTA)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Fatiador de texto
Settings.node_parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print("✅ LlamaIndex configurado perfeitamente! (Gemini para LLM + Local para Embeddings)")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ LlamaIndex configurado perfeitamente! (Gemini para LLM + Local para Embeddings)


## 3. Selecionar o PDF
1. Abra a aba **📁 Arquivos** na barra lateral esquerda do Colab.
2. Arraste o PDF para a pasta raiz (`/content`) e espere o upload terminar.
3. Rode a célula abaixo. Se houver mais de um PDF, preencha `PDF_NOME` com o nome do arquivo desejado.

In [12]:
PDF_NOME = ''  # opcional: preencha se quiser um arquivo específico (ex: 'dnd.pdf')

# Procura recursivamente por qualquer PDF dentro de /content e subpastas
base_dir = Path('/content')
pdfs = sorted(list(base_dir.rglob('*.pdf')))

print('🔍 Buscando PDFs no Colab...')
if not pdfs:
    print('\n❌ Nenhum arquivo .pdf encontrado!')
    print('Arquivos atualmente presentes em /content:')
    for item in base_dir.iterdir():
        print(f'  - {item.name} {"(pasta)" if item.is_dir() else ""}')
    raise FileNotFoundError(
        'Nenhum PDF encontrado em /content. '
        'Envie o arquivo arrastando para a aba 📁 Arquivos na barra lateral esquerda e tente novamente.'
    )

print(f'\n📚 Encontrado(s) {len(pdfs)} PDF(s):')
for i, p in enumerate(pdfs, start=1):
    caminho_rel = p.relative_to(base_dir)
    print(f'  {i}. {caminho_rel} ({p.stat().st_size / 1e6:.1f} MB)')

if PDF_NOME:
    matches = [p for p in pdfs if p.name.lower() == PDF_NOME.lower()]
    if not matches:
        raise FileNotFoundError(f'"{PDF_NOME}" não encontrado na lista acima.')
    PDF_PATH = matches[0]
elif len(pdfs) == 1:
    PDF_PATH = pdfs[0]
else:
    # Se houver mais de um, escolhe o primeiro por padrão ou avisa para especificar
    print(f'\n⚠️ Mais de um PDF encontrado. Usando por padrão o primeiro: {pdfs[0].name}')
    print('Caso queira outro, preencha PDF_NOME = "nome_do_arquivo.pdf" e rode novamente.')
    PDF_PATH = pdfs[0]

print(f'\n✅ PDF selecionado: {PDF_PATH}')

🔍 Buscando PDFs no Colab...

📚 Encontrado(s) 1 PDF(s):
  1. dampd-5e---livro-do-jogador-2024.pdf (36.7 MB)

✅ PDF selecionado: /content/dampd-5e---livro-do-jogador-2024.pdf


## 4. Extrair e limpar o texto (uma página = um documento)
Usamos o **PyMuPDF**, que lida bem com layouts em colunas. Cada página guarda seu número nos metadados, para que as respostas possam citar a página.

In [13]:
def limpar_texto(texto: str) -> str:
    texto = re.sub(r'(\w)-\n(\w)', r'\1\2', texto)  # Junta palavras quebradas
    # Remove linhas de artistas/créditos comuns em fim de página
    texto = re.sub(r'(?i)\b(ilustra[çc][ãa]o|créditos|copyright|craig j spearing|caio monteiro)\b.*', '', texto)
    # Remove cabeçalhos repetitivos como "CAPÍTULO X | ..."
    texto = re.sub(r'CAPÍTULO\s+\d+\s*\|\s*[^\n]+', '', texto)
    texto = re.sub(r'[ \t]+', ' ', texto)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()

nome_livro = Path(PDF_PATH).stem
paginas = PyMuPDFReader().load_data(file_path=Path(PDF_PATH))

documentos = []
for i, pag in enumerate(paginas, start=1):
    texto = limpar_texto(pag.text)
    if len(texto) < 30:  # ignora páginas vazias / só com ilustração
        continue
    documentos.append(Document(text=texto, metadata={'livro': nome_livro, 'pagina': i}))

print(f'📚 {len(paginas)} páginas lidas, {len(documentos)} com texto útil')
print('\n--- Amostra da página', documentos[0].metadata['pagina'], '---')
print(documentos[0].text[:800])

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


📚 397 páginas lidas, 388 com texto útil

--- Amostra da página 1 ---
O MAIOR JOGO DE RPG DO MUNDO
LIVRO DO JOGADOR
LIVRO DO JOGADOR


## 5. Quebrar em trechos e criar o índice vetorial
Na primeira execução, o livro é dividido em chunks e indexado (pode levar alguns minutos). Enquanto a sessão do Colab estiver ativa, as próximas execuções carregam o índice de `/content/rpg_indices`.

> 💡 O índice é apagado quando a sessão reinicia. Para reaproveitá-lo, baixe a pasta `rpg_indices` pela aba Arquivos e envie de volta para `/content` na próxima sessão.

In [14]:
pasta_indice = Path(PERSIST_DIR) / nome_livro
RECRIAR_INDICE = False  # Mude para True caso altere o CHUNK_SIZE ou o arquivo PDF

# Se o índice já foi gerado nesta sessão, apenas carrega do disco instantaneamente
if (pasta_indice / 'docstore.json').exists() and not RECRIAR_INDICE:
    print('📂 Carregando índice salvo de:', pasta_indice)
    storage = StorageContext.from_defaults(persist_dir=str(pasta_indice))
    index = load_index_from_storage(storage)
else:
    # Caso contrário, fatia o livro e gera os vetores usando o Settings.embed_model configurado na Célula 2
    nodes = Settings.node_parser.get_nodes_from_documents(documentos, show_progress=True)
    print(f'✂️ Livro dividido em {len(nodes)} trechos.')
    print('🧠 Gerando embeddings locais...')

    index = VectorStoreIndex(nodes, show_progress=True)

    pasta_indice.mkdir(parents=True, exist_ok=True)
    index.storage_context.persist(persist_dir=str(pasta_indice))
    print(f'💾 Concluído! Índice salvo em: {pasta_indice}')

Parsing nodes:   0%|          | 0/388 [00:00<?, ?it/s]

✂️ Livro dividido em 1162 trechos.
🧠 Gerando embeddings locais...


Generating embeddings:   0%|          | 0/1162 [00:00<?, ?it/s]

💾 Concluído! Índice salvo em: /content/rpg_indices/dampd-5e---livro-do-jogador-2024


## 6. Motor de perguntas e respostas
Prompt em português, instruindo o modelo a usar **apenas** o conteúdo do livro e citar as páginas.

In [16]:

QA_PROMPT = PromptTemplate(
    "Você é um consultor especialista no livro de RPG fornecido.\n"
    "Responda à pergunta em português com precisão, usando o contexto abaixo.\n\n"
    "### REGRAS OBRIGATÓRIAS DE FORMATAÇÃO:\n"
    "1. **SEMPRE QUE FOR PEDIDO UMA LISTA OU TABELA:**\n"
    "   - Estruture a resposta preferencialmente em **Tabela Markdown** (`| Coluna 1 | Coluna 2 | ... |`).\n"
    "   - **Caso 1 (Tabelas do livro, ex: Talentos):** Se o contexto contiver uma tabela ou catálogo, replique as colunas originais (ex: Nome, Categoria, Requisitos) de forma idêntica.\n"
    "   - **Caso 2 (Propriedades e mecânicas, ex: Maestria de Armas):** Se os itens estiverem descritos em texto corrido, extraia cada um deles e monte uma tabela com: `| Propriedade / Item | Resumo da Regra / Efeito | Página |`.\n"
    "2. **DESTACAR DADOS E NÚMEROS:**\n"
    "   - Sempre coloque dados (ex: **1d20**, **2d6**), CDs, bônus e condições em **negrito**.\n"
    "3. **DIRETO AO PONTO:**\n"
    "   - Não faça introduções desnecessárias. Vá direto para a tabela ou lista solicitada.\n\n"
    "Contexto do livro:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Pergunta: {query_str}\n"
    "Resposta estruturada:"
)

# Mantemos a busca eficiente e sem excesso de tokens
query_engine = index.as_query_engine(
    similarity_top_k=TOP_K,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.52)],
    text_qa_template=QA_PROMPT,
    response_mode="simple_summarize"
)

async def perguntar(pergunta: str, mostrar_paginas: bool = True):
    resp = await query_engine.aquery(pergunta)
    display(Markdown(f"### 🎲 {pergunta}\n\n{resp.response}"))
    if mostrar_paginas:
        pags = sorted({n.metadata.get('pagina') for n in resp.source_nodes if n.metadata.get('pagina')})
        if pags:
            display(Markdown(f"\n<sub>📖 *Páginas consultadas: {', '.join(map(str, pags))}*</sub>"))

print("✅ Prompt aprimorado para tabelas e listas!")

✅ Prompt aprimorado para tabelas e listas!


### Exemplos de perguntas

In [17]:
await perguntar('LiListe os talentos de origem')

### 🎲 LiListe os talentos de origem

Empty Response


<sub>📖 *Páginas consultadas: 20, 39, 42, 185, 186, 189, 205, 207, 212*</sub>

In [ ]:
await perguntar('Monte uma tabela com todos os talentos da categoria Origem presentes no livro.')

## 7. Apenas buscar trechos (sem gerar resposta)
Útil para conferir onde uma regra aparece no livro, sem gastar chamadas do LLM.

In [ ]:
retriever = index.as_retriever(similarity_top_k=TOP_K)

def buscar_trechos(consulta: str, k: int = TOP_K):
    retriever.similarity_top_k = k
    for n in retriever.retrieve(consulta):
        print(f"\n=== p. {n.metadata.get('pagina')} | similaridade {n.score:.3f} ===")
        print(n.node.get_content()[:700])

buscar_trechos('magias de cura', k=3)

## 8. Chat interativo com memória
O chat lembra das perguntas anteriores (ex.: *"E para o mago?"* depois de perguntar sobre o guerreiro). Digite `sair` para encerrar.

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

chat_engine = index.as_chat_engine(
    chat_mode='condense_plus_context',
    memory=ChatMemoryBuffer.from_defaults(token_limit=4000),
    similarity_top_k=TOP_K,
    system_prompt=(
        'Você é um mestre de RPG experiente que conhece profundamente o livro fornecido. '
        'Responda em português, usando apenas o conteúdo do livro e citando as páginas (p. X). '
        'Se algo não estiver no livro, diga isso claramente.'
    ),
)

print('🎲 Chat iniciado! Digite "sair" para encerrar ou "limpar" para zerar a memória.\n')
while True:
    pergunta = input('Você: ').strip()
    if pergunta.lower() in ('sair', 'exit', 'quit', ''):
        print('Até a próxima sessão! 🐉')
        break
    if pergunta.lower() == 'limpar':
        chat_engine.reset()
        print('🧹 Memória limpa.\n')
        continue
    resp = chat_engine.chat(pergunta)
    paginas_usadas = sorted({n.metadata.get('pagina') for n in resp.source_nodes})
    display(Markdown(f'**Mestre:** {resp.response}\n\n<sub>📖 páginas consultadas: {paginas_usadas}</sub>'))

## 💡 Dicas de ajuste

| Problema | O que ajustar |
|---|---|
| Respostas incompletas (regras espalhadas) | Aumente `TOP_K` (8–10) |
| Trechos cortando tabelas/regras no meio | Aumente `CHUNK_SIZE` (1500–2048) e `CHUNK_OVERLAP` |
| Respostas trazendo assuntos misturados | Diminua `CHUNK_SIZE` (512) |
| Texto de colunas embaralhado | Troque o leitor por `docling` ou `marker` para extrair em Markdown |
| Erro 429 (limite de requisições) na indexação | Diminua `embed_batch_size` e rode a célula de novo |

Depois de mudar `CHUNK_SIZE`/`CHUNK_OVERLAP`, rode a célula 5 com `RECRIAR_INDICE = True`.